# v3 비교 실험 — 같은 100명에게 voter_context_v3 + system_prompt_v3 적용

- 입력: `vote_results_all.csv` 에서 `model == openai/gpt-5.4-mini` AND `voter_context_version == v2` 페르소나 100명
- 처리: 동일한 OpenAI 모델·동일 페르소나에 v3 컨텍스트(지역·세대 섹션 제거)로 투표 재추론
- 결과: `vote_results_all.csv` 에 누적 (version=v3), `response/` 에 raw 저장
- political_interest 는 **이번에는 사용하지 않음**

In [1]:
# [Cell 1] imports & env
import os, json, time, re
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

BASE = Path.cwd() if Path.cwd().name == "backend" else Path.cwd() / "backend"
CONTEXT_DIR = BASE / "context"
RESULTS_PATH = BASE / "vote_results_all.csv"
RESPONSE_DIR = BASE / "response"
RESPONSE_DIR.mkdir(exist_ok=True)
LOG_DIR = BASE / "log"
LOG_DIR.mkdir(exist_ok=True)

MODEL = "gpt-5.4-mini"
VERSION = "v3"

for env_path in [BASE / ".env.local", BASE.parent / ".env.local"]:
    if env_path.exists():
        load_dotenv(env_path)
        print(f"loaded: {env_path}")
        break
assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY 필요"

def safe_filename(s: str) -> str:
    return re.sub(r"[^\w\-.]", "_", s)

print(f"model: {MODEL}, version: {VERSION}")

loaded: z:\Github\persona-million\.env.local
model: gpt-5.4-mini, version: v3


In [2]:
# [Cell 2] v2 + gpt-5.4-mini 페르소나 100명 로드
df_all = pd.read_csv(RESULTS_PATH, encoding="utf-8-sig")
mask = (df_all["model"] == f"openai/{MODEL}") & (df_all["voter_context_version"] == "v2")
targets = df_all[mask].drop_duplicates(subset=["persona_uuid"]).reset_index(drop=True)
print(f"target personas: {len(targets)}")

# 이미 v3로 처리된 uuid 는 건너뛰기 (재실행 안전)
done_v3 = set(df_all[(df_all["model"] == f"openai/{MODEL}") & (df_all["voter_context_version"] == "v3")]["persona_uuid"].tolist())
todo = targets[~targets["persona_uuid"].isin(done_v3)].reset_index(drop=True)
print(f"이미 v3 처리됨: {len(done_v3)}, 남은 작업: {len(todo)}")

target personas: 100
이미 v3 처리됨: 0, 남은 작업: 100


In [3]:
# [Cell 3] v3 프롬프트 + 체인 + 파서
def load_context(version: str) -> str:
    return (CONTEXT_DIR / f"voter_context_{version}.md").read_text(encoding="utf-8")
def load_system_prompt(version: str) -> str:
    return (CONTEXT_DIR / f"system_prompt_{version}.md").read_text(encoding="utf-8")

POLITICAL_CONTEXT = load_context(VERSION)
SYSTEM_TEMPLATE = load_system_prompt(VERSION)
print(f"voter_context_{VERSION}.md: {len(POLITICAL_CONTEXT)} chars")
print(f"system_prompt_{VERSION}.md: {len(SYSTEM_TEMPLATE)} chars")

USER_TEMPLATE = """다음 페르소나가 2026년 6월 3일 지방선거(또는 가까운 미래의 총선·대선)에서 어느 정당을 지지·투표할지 추론하시오.

{persona_card}

JSON만 출력."""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_TEMPLATE),
    ("user", USER_TEMPLATE),
])
llm = ChatOpenAI(
    model=MODEL, temperature=0.7,
    model_kwargs={"response_format": {"type": "json_object"}},
)
chain = prompt | llm

VOTE_KEYS = ("vote", "party", "정당", "지지정당", "투표", "선택", "지지")
REASON_KEYS = ("reason", "이유", "rationale", "사유", "근거", "설명")

def _try_complete_json(s: str) -> str:
    last_close = s.rfind("}")
    last_open = s.rfind("{")
    if last_open == -1: return s
    if last_close > last_open: return s[last_open:last_close + 1]
    candidate = s[last_open:].rstrip().rstrip(",")
    if candidate.count('"') % 2 == 1: candidate += '"'
    candidate += "}"
    return candidate

def parse_response(raw: str) -> dict:
    s = re.sub(r"<think>.*?</think>", "", raw, flags=re.DOTALL)
    s = re.sub(r"```(?:json)?\s*", "", s).replace("```", "")
    m = re.search(r"\{.*\}", s, flags=re.DOTALL)
    if m:
        try: return json.loads(m.group(0))
        except json.JSONDecodeError: pass
    try: return json.loads(_try_complete_json(s))
    except json.JSONDecodeError:
        raise ValueError(f"no parseable JSON: {raw[:200]!r}")

def pick(d, keys):
    for k in keys:
        v = d.get(k)
        if v: return v
    return None

def persona_card(r) -> str:
    fields = [
        ("성별", r["sex"]), ("연령", f"{r['age']}세"),
        ("혼인상태", r["marital_status"]),
        ("가구형태", r["family_type"]), ("주거", r["housing_type"]),
        ("학력", r["education_level"]), ("전공", r["bachelors_field"]),
        ("직업", r["occupation"]),
        ("지역", f"{r['province']} {r['district']}"),
    ]
    demo = "\n".join(f"- {k}: {v}" for k, v in fields if v)
    narr = (
        f"[요약]\n{r['persona_summary']}\n\n"
        f"[직업적 면모]\n{r['professional_persona']}\n\n"
        f"[가족 면모]\n{r['family_persona']}\n\n"
        f"[문화적 배경]\n{r['cultural_background']}\n\n"
        f"[관심사]\n{r['hobbies_and_interests']}\n\n"
        f"[목표]\n{r['career_goals_and_ambitions']}"
    )
    return f"## 인구통계\n{demo}\n\n## 인물 서사\n{narr}"

voter_context_v3.md: 2400 chars
system_prompt_v3.md: 1122 chars


In [4]:
# [Cell 4] 100명 v3 추론 → CSV 누적 + response/ 저장
log_path = LOG_DIR / f"{time.strftime('%Y%m%d-%H%M%S')}_voteV3_openai_{safe_filename(MODEL)}_{VERSION}.log"
log_f = log_path.open("w", encoding="utf-8")
def log(msg):
    print(msg, flush=True)
    log_f.write(msg + "\n"); log_f.flush()

log(f"=== openai/{MODEL} × {len(todo)} (version={VERSION}) ===")
success = 0
for i, r in todo.iterrows():
    t0 = time.perf_counter()
    raw = ""
    try:
        msg = chain.invoke({"political_context": POLITICAL_CONTEXT, "persona_card": persona_card(r)})
        raw = msg.content if hasattr(msg, "content") else str(msg)
        p = parse_response(raw)
        vote = pick(p, VOTE_KEYS)
        reason = pick(p, REASON_KEYS)
        if not vote or not reason:
            raise ValueError(f"missing keys: {list(p.keys())}")
    except Exception as e:
        el = time.perf_counter() - t0
        log(f"[{i+1:>3}/{len(todo)}] ERROR ({el:.1f}s): {str(e)[:120]}")
        ts = time.strftime("%Y%m%d-%H%M%S")
        fname = f"FAIL_{ts}_openai_{safe_filename(MODEL)}_{r['persona_uuid'][:8]}_{VERSION}.txt"
        (RESPONSE_DIR / fname).write_text(
            f"=== ERROR: {e}\n=== MODEL: openai/{MODEL}\n=== UUID: {r['persona_uuid']}\n\n{raw}",
            encoding="utf-8")
        continue
    el = time.perf_counter() - t0
    success += 1

    ts = time.strftime("%Y%m%d-%H%M%S")
    fname = f"{ts}_openai_{safe_filename(MODEL)}_{r['persona_uuid'][:8]}_{VERSION}_{VERSION}.json"
    out = {
        "timestamp": ts,
        "model": f"openai/{MODEL}",
        "voter_context_version": VERSION,
        "system_prompt_version": VERSION,
        "persona_uuid": r["persona_uuid"],
        "elapsed_sec": round(el, 3),
        "persona": {
            "sex": r["sex"], "age": int(r["age"]),
            "marital_status": r["marital_status"],
            "family_type": r["family_type"],
            "housing_type": r["housing_type"],
            "education_level": r["education_level"],
            "bachelors_field": r["bachelors_field"],
            "occupation": r["occupation"],
            "province": r["province"], "district": r["district"],
            "persona_summary": r["persona_summary"],
        },
        "raw_response": raw,
        "parsed": {"vote": vote, "reason": reason},
    }
    (RESPONSE_DIR / fname).write_text(json.dumps(out, ensure_ascii=False, indent=2), encoding="utf-8")

    rec = {
        "persona_uuid": r["persona_uuid"],
        "sex": r["sex"], "age": int(r["age"]),
        "marital_status": r["marital_status"],
        "family_type": r["family_type"], "housing_type": r["housing_type"],
        "education_level": r["education_level"],
        "bachelors_field": r["bachelors_field"],
        "occupation": r["occupation"],
        "province": r["province"], "district": r["district"],
        "persona_summary": r["persona_summary"],
        "vote": vote, "reason": reason,
        "model": f"openai/{MODEL}",
        "elapsed_sec": round(el, 3),
        "response_file": fname,
        "voter_context_version": VERSION,
        "system_prompt_version": VERSION,
        "professional_persona": r["professional_persona"],
        "sports_persona": r["sports_persona"],
        "arts_persona": r["arts_persona"],
        "travel_persona": r["travel_persona"],
        "culinary_persona": r["culinary_persona"],
        "family_persona": r["family_persona"],
        "cultural_background": r["cultural_background"],
        "skills_and_expertise": r["skills_and_expertise"],
        "hobbies_and_interests": r["hobbies_and_interests"],
        "career_goals_and_ambitions": r["career_goals_and_ambitions"],
    }
    df_rec = pd.DataFrame([rec])
    if RESULTS_PATH.exists():
        df_rec.to_csv(RESULTS_PATH, mode="a", header=False, index=False, encoding="utf-8-sig")
    else:
        df_rec.to_csv(RESULTS_PATH, index=False, encoding="utf-8-sig")
    log(f"[{i+1:>3}/{len(todo)}] {r['sex']} {r['age']}세 {r['province']:<6} "
        f"{str(r['occupation'])[:14]:<14} → {vote[:10]:<10} ({el:.1f}s)")

log(f"\n결과: {success}/{len(todo)} 성공")
log_f.close()

=== openai/gpt-5.4-mini × 100 (version=v3) ===
[  1/100] 여자 79세 충청남    건물 청소원         → 더불어민주당     (3.4s)
[  2/100] 남자 29세 대구     무직             → 국민의힘       (2.3s)
[  3/100] 남자 27세 서울     전직 금속제품 생산 관리자 → 개혁신당       (2.3s)
[  4/100] 여자 71세 대전     무직             → 더불어민주당     (3.5s)
[  5/100] 여자 52세 부산     전동차 정비원        → 국민의힘       (2.1s)
[  6/100] 여자 20세 서울     무직             → 개혁신당       (2.2s)
[  7/100] 여자 26세 제주     초등학교 교사        → 더불어민주당     (2.1s)
[  8/100] 남자 45세 경상남    용접기 조작원        → 국민의힘       (2.2s)
[  9/100] 여자 63세 충청북    요양 보호사         → 더불어민주당     (2.4s)
[ 10/100] 여자 67세 경기     무직             → 더불어민주당     (1.9s)
[ 11/100] 여자 40세 강원     혼례 종사원         → 더불어민주당     (2.5s)
[ 12/100] 남자 48세 서울     마케팅 전문가        → 더불어민주당     (2.4s)
[ 13/100] 남자 45세 인천     그 외 택배원        → 더불어민주당     (1.9s)
[ 14/100] 남자 38세 충청남    한식 조리사         → 더불어민주당     (4.5s)
[ 15/100] 여자 26세 서울     보육교사           → 더불어민주당     (2.3s)
[ 16/100] 남자 53세 부산     대형 화물차 운전원     → 국민의힘       (2.9s)
[ 17/100]

In [5]:
# [Cell 5] v2 vs v3 비교 — 같은 100명에서 어느 정당이 어떻게 변했나
df_all = pd.read_csv(RESULTS_PATH, encoding="utf-8-sig")
v2 = df_all[(df_all["model"] == f"openai/{MODEL}") & (df_all["voter_context_version"] == "v2")][["persona_uuid", "vote"]].drop_duplicates("persona_uuid")
v3 = df_all[(df_all["model"] == f"openai/{MODEL}") & (df_all["voter_context_version"] == "v3")][["persona_uuid", "vote"]].drop_duplicates("persona_uuid")
merged = v2.merge(v3, on="persona_uuid", suffixes=("_v2", "_v3"))
print(f"비교 가능 페르소나: {len(merged)}명\n")

print("=== v2 분포 ===")
print(merged["vote_v2"].value_counts())
print("\n=== v3 분포 ===")
print(merged["vote_v3"].value_counts())

merged["changed"] = merged["vote_v2"] != merged["vote_v3"]
print(f"\n선택이 바뀐 사람: {merged['changed'].sum()}/{len(merged)} ({merged['changed'].mean()*100:.1f}%)")

print("\n=== 전이행렬 (v2 → v3) ===")
ct = pd.crosstab(merged["vote_v2"], merged["vote_v3"], margins=True)
print(ct)

비교 가능 페르소나: 100명

=== v2 분포 ===
vote_v2
국민의힘      42
더불어민주당    40
무당층/기권    10
개혁신당       8
Name: count, dtype: int64

=== v3 분포 ===
vote_v3
더불어민주당    49
국민의힘      37
개혁신당       9
무당층/기권     5
Name: count, dtype: int64

선택이 바뀐 사람: 29/100 (29.0%)

=== 전이행렬 (v2 → v3) ===
vote_v3  개혁신당  국민의힘  더불어민주당  무당층/기권  All
vote_v2                                 
개혁신당        5     2       1       0    8
국민의힘        0    30      11       1   42
더불어민주당      1     5      33       1   40
무당층/기권      3     0       4       3   10
All         9    37      49       5  100
